# CellLM Chat — Chua–Yang + Oja memory

Trains a small English conversational CellLM. Durable checkpoints, tokenizer, metrics, and generated samples are written to the submitter output mount.

In [ ]:
%pip install git+https://github.com/tomieiro/libPyCelNN.git
%pip install -r requirements.txt
%pip install -e . --no-deps

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys
import torch

root = Path('/workspace')
outputs = root / 'outputs' / 'chat'
outputs.mkdir(parents=True, exist_ok=True)

resume = root / 'resume' / 'progress.pt'
local_tokenizer = root / 'resume' / 'tokenizer.json'
if local_tokenizer.exists() and not (outputs / 'tokenizer.json').exists():
    shutil.copy2(local_tokenizer, outputs / 'tokenizer.json')

assert torch.cuda.is_available(), 'the submitted job did not receive a GPU'
print('GPU:', torch.cuda.get_device_name(0), flush=True)

command = [
    sys.executable, 'train_chat.py',
    '--data',
    str(root / 'data' / 'simple-dialogues.jsonl'),
    str(root / 'data' / 'smoltalk-everyday.jsonl'),
    '--output', str(outputs),
    '--steps', '20000',
    '--batch-size', '128',
    '--sequence-length', '128',
    '--context', '16',
    '--dimensions', '256',
    '--radius', '1',
    '--dynamics-steps', '15',
    '--vocab-size', '1024',
    '--rule', 'oja',
    '--device', 'cuda',
]
if resume.exists():
    staged = outputs / 'progress.pt'
    shutil.copy2(resume, staged)
    command.extend(['--resume', str(staged)])
print(' '.join(command), flush=True)
subprocess.run(command, check=True)

In [ ]:
evaluation = [
    sys.executable, 'evaluate_chat.py',
    str(outputs / 'final.pt'),
    '--tokenizer', str(outputs / 'tokenizer.json'),
    '--output', str(outputs / 'evaluation.json'),
    '--device', 'cuda',
]
print(' '.join(evaluation), flush=True)
subprocess.run(evaluation, check=True)